## Tymoteusz Bultrowicz - second task report

### Approach

In my work I analyzed and replicated approach of a top competitor team in IEEE Big Data Cup 2022 - \
    following steps in work of Vrizlynn L. L. Thing. His workflow with processing images can be described as follows:

1. Prepare negative instances by randomly shuffling pairs given in train dataset, so in total we have 20000 instances.
2. Split them into train and test set, cut out small part of test set to validate the progress.
3. For main model use ResNet152 model from PyTorch, adapted with:
    : Six channels of input layer instead of three, as we work with pairs of pictures.
    : Switching number of output nodes to only one for binary classification.
4. For model to work better on low quality data and have artificially extended train set, team used data augmentation of pictures. \
Precise description of techniques used further.

### My adjustments
1. I decided to split photos before, and then add negative instances. It asserts that CNN never sees the picture before being tested on it.
2. When doing basic first runs of task I realized that lighter model of ResNet18 also does great job for this task. That's why \
I switched to lighter model for quicker compute times, because my workstation doesn't have Nvidia GPU with CUDA.

### Comments on my work
As we have no experience working with pytorch library on labs, I admit I had to support myself with Gemini in some parts of code, \
they will be marked below. Besides that it also helped me prepare smart dataset class, which prevents me from having to load \
every single photo into RAM at once, when training the model.

### Execute time
As model very quickly adapted to data when trained with adam optimizer, not many epochs were needed. It allowed computations \
to finish in around ~50 minutes for 5 epochs. Along with other parts of code, execute time was around 55mins.

### Preparing 'links' datatables

In [1]:
from PIL import Image
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
import torch
import sys

RSEED = 15968

np.random.seed(RSEED)
torch.manual_seed(RSEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RSEED)

# loading data
train_data = pd.read_csv('data/train/train.csv')
train_data = train_data.drop(columns=['id'])
train_data['class'] = 1

# splitting true instances of pictures between train, test and validation set
train_true, val_true = train_test_split(train_data, test_size=0.2, random_state=RSEED)
val_true, test_true = train_test_split(val_true, test_size=0.75, random_state=RSEED)

# resetting indexes
train_true = train_true.reset_index(drop=True)
val_true = val_true.reset_index(drop=True)
test_true = test_true.reset_index(drop=True)

def generate_false_pairs(df):
    size = len(df)
    false_pairs = []
    for i in range(size):
        # randomize shift for negative instances
        rand_ind = np.random.randint(1, size - 1)
        false_pairs.append({
            "input_path": df.iloc[i, 0],
            "encoded_path": df.iloc[(i + rand_ind) % size, 1],
            "class": 0
        })
    return pd.DataFrame(false_pairs)

# adding false instances to train, test and validation set
final_train = pd.concat([train_true, generate_false_pairs(train_true)], ignore_index=True)
final_val = pd.concat([val_true, generate_false_pairs(val_true)], ignore_index=True)
final_test = pd.concat([test_true, generate_false_pairs(test_true)], ignore_index=True)

# splitting into X and y
X_train = final_train[['input_path', 'encoded_path']].values
y_train = final_train['class'].values
X_val = final_val[['input_path', 'encoded_path']].values
y_val = final_val['class'].values
X_test = final_test[['input_path', 'encoded_path']].values
y_test = final_test['class'].values

# checking if we shapes are as expected => test: 16000x2, val: 1000x2, test: 3000x2
print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape, y_test.shape)

(16000, 2) (16000,)
(1000, 2) (1000,)
(3000, 2) (3000,)


### Smart dataset that allows to train model loading only one batch of pictures into RAM at once.

In [2]:
from torch.utils.data import Dataset

# Helper function to load an image and convert it to a numpy array.
def load_image(image_path):
    try:
        image = Image.open(image_path).convert('RGB')
        return np.array(image)
    except Exception as e:
        print(f"Error loading image: {e}")
        sys.exit(1)

# Custom Dataset class to handle pairs of images as their labels.
class PathImagePairDataset(Dataset):
    def __init__(self, X, y, transform=None):
        """
        X: Tablica/lista ścieżek, np. [['img1_orig.jpg', 'img1_enc.jpg'], ...]
        y: Tablica/lista etykiet, np. [1, 0, 1, ...]
        """
        self.X = X
        self.y = y
        self.transform = transform

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        path_original, path_encoded = self.X[idx]
        label = self.y[idx]
        
        img_original = load_image(path_original)
        img_encoded = load_image(path_encoded)
        
        if self.transform:
            augmented = self.transform(image=img_original, encoded_image=img_encoded)
            img_original_t = augmented['image']          
            img_encoded_t = augmented['encoded_image']   
        else:
            img_original_t = torch.tensor(img_original).permute(2, 0, 1).float()
            img_encoded_t = torch.tensor(img_encoded).permute(2, 0, 1).float()

        combined_tensor = torch.cat((img_original_t, img_encoded_t), dim=0)
        
        return combined_tensor, torch.tensor(label, dtype=torch.float32)

### Data augmentation

Pipeline for augmenting the data follows techniques mention in paper which is:
1. ShiftScaleRotate,
2. HorizontalFlip,
3. RandomBrightnessContrast,
4. MotionBlur,
5. GaussNoise,
6. ToGray

During my research I found that library **albumentations** contains all of this operations \
implemented, in very easy to use way, highly compatible with pyTorch, I decided to use this tool.

Besides this transformations I also resized data for quicker execute time and normalised the data, as \
ResNet is best suitable for use on normalised photos.


In [3]:
import albumentations as alb
from albumentations.pytorch import ToTensorV2

augment_transform = alb.Compose([
    alb.Resize(224, 224), 
    alb.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.1, rotate_limit=30, p=0.5),
    alb.HorizontalFlip(p=0.5),
    alb.RandomBrightnessContrast(p=0.2),
    alb.MotionBlur(blur_limit=3, p=0.2),
    alb.GaussNoise(p=0.2),
    alb.ToGray(p=0.2),
    alb.ImageCompression(quality_range=(60, 100), p=0.2),
    alb.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
], additional_targets={'encoded_image': 'image'})

normal_transform = alb.Compose([
    alb.Resize(224, 224),
    alb.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
], additional_targets={'encoded_image': 'image'})

# Utworzenie datasetów z użycie
train_dataset = PathImagePairDataset(X=X_train, y=y_train, transform=augment_transform)
val_dataset = PathImagePairDataset(X=X_val, y=y_val, transform=normal_transform)
test_dataset = PathImagePairDataset(X=X_test, y=y_test, transform=normal_transform)

/home/tymmeek/Desktop/2yINF/MachineLearning/projects/classification/.venv/lib/python3.13/site-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


### Training phase

As mentioned before we use ResNet18 modified to work with pair of pictures for binary classification. \
Due to fact that model adapts very quickly, only small amount of 5 epochs is used.

### Loss function

For the loss function I used BCEWithLogitsLoss, because it is standard and most stable tool for binary \
classification tasks in PyTorch.

In [4]:

import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
from torch.utils.data import DataLoader
from tqdm import tqdm

# Modifying ResNet18 for our usecase. Help of Gemini was used to create this function.
def get_modified_resnet18():
    weights = models.ResNet18_Weights.DEFAULT
    model = models.resnet18(weights=weights)
    
    old_conv = model.conv1
    model.conv1 = nn.Conv2d(
        6, 
        old_conv.out_channels, 
        kernel_size=old_conv.kernel_size, 
        stride=old_conv.stride, 
        padding=old_conv.padding, 
        bias=False
    )
    with torch.no_grad():
        model.conv1.weight[:, :3, :, :] = old_conv.weight
        model.conv1.weight[:, 3:, :, :] = old_conv.weight
        
    model.fc = nn.Linear(model.fc.in_features, 1)
    return model

# Function to train the model. Help of Gemini was used to create this function,
# mainly with handling optimizer logic and collecting+printing metrics during training and validation.
def train_model(model, train_loader, val_loader, num_epochs=5, lr=0.001):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Begin training on: {device}")
    model = model.to(device)
    
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    
    for epoch in range(num_epochs):
        # Single epoch training loop (500 x 32 batches)
        model.train()
        train_loss, train_correct, train_total = 0.0, 0, 0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]")
        for inputs, labels in pbar:
            inputs, labels = inputs.to(device), labels.to(device)
            labels = labels.unsqueeze(1)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * inputs.size(0)
            preds = torch.sigmoid(outputs) >= 0.5
            train_correct += (preds == labels).sum().item()
            train_total += labels.size(0)
            
            pbar.set_postfix({'loss': loss.item()})
        
        # Testing results of CNN after epoch.
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                labels = labels.unsqueeze(1)
                
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item() * inputs.size(0)
                preds = torch.sigmoid(outputs) >= 0.5
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)
                
        val_acc = val_correct / val_total
        val_loss = val_loss / val_total
        train_acc = train_correct / train_total
        train_loss = train_loss / train_total
        
        print(f"Epoch {epoch+1} Summary:")
        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}\n")

# Creating DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

model = get_modified_resnet18()
train_model(model, train_loader, val_loader, num_epochs=4)

Begin training on: cpu


Epoch 1/4 [Train]: 100%|██████████| 500/500 [09:13<00:00,  1.11s/it, loss=0.0731] 


Epoch 1 Summary:
Train Loss: 0.1522 | Train Acc: 0.9318
Val Loss: 0.1119 | Val Acc: 0.9650



Epoch 2/4 [Train]: 100%|██████████| 500/500 [09:06<00:00,  1.09s/it, loss=0.00294] 


Epoch 2 Summary:
Train Loss: 0.0411 | Train Acc: 0.9849
Val Loss: 0.0319 | Val Acc: 0.9860



Epoch 3/4 [Train]: 100%|██████████| 500/500 [09:03<00:00,  1.09s/it, loss=0.0228]  


Epoch 3 Summary:
Train Loss: 0.0365 | Train Acc: 0.9874
Val Loss: 0.0278 | Val Acc: 0.9870



Epoch 4/4 [Train]: 100%|██████████| 500/500 [09:02<00:00,  1.08s/it, loss=0.00568] 


Epoch 4 Summary:
Train Loss: 0.0331 | Train Acc: 0.9883
Val Loss: 0.0085 | Val Acc: 0.9970



### Testing - see how our model performs

In [5]:
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.BCEWithLogitsLoss()

model = model.to(device)
model.eval()

test_loss, test_correct, test_total = 0.0, 0, 0
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        labels = labels.unsqueeze(1)
        
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        test_loss += loss.item() * inputs.size(0)
        preds = torch.sigmoid(outputs) >= 0.5
        test_correct += (preds == labels).sum().item()
        test_total += labels.size(0)

test_acc = test_correct / test_total
test_loss = test_loss / test_total

print("Result achieved on test set by final model:")
print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}\n")

Result achieved on test set by final model:
Test Loss: 0.0058 | Test Acc: 0.9977



### Summary

As very high accuracy of 99+% was achieved on test set, we can happily \
finish work, without switching into heavier model of ResNet152 or trying different approaches.

In [6]:
# Generating predictions for the provided test set and saving them to a text file.

result_dbase = pd.read_csv('data/test/test.csv')
result_dbase = result_dbase.drop(columns=['id'])

results_dataset = PathImagePairDataset(
    X=result_dbase.values, 
    y=np.zeros(len(result_dbase)), 
    transform=normal_transform
)
results_loader = DataLoader(results_dataset, batch_size=32, shuffle=False, num_workers=2)

all_outs = []
with torch.no_grad():
    for inputs, _ in results_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        preds = (torch.sigmoid(outputs) >= 0.5).int()
        for p in preds:
            all_outs.append(str(p.item()))

with open('Tymoteusz_Bultrowicz_decisions.txt', 'w') as f:
    f.write('\n'.join(all_outs))

### Links to used libraries / methods

1. Albumentation library: https://albumentations.ai/
2. Pytorch models: https://docs.pytorch.org/vision/main/models.html
3. Pillow - image loading: https://pillow.readthedocs.io/en/stable/
4. Train test split: https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html